# Graph Classification with TopKPooling on Colors Dataset

Graph Classification on TUDataset (Colors): Hierarchical graph representation learning using TopKPooling. This notebook implements the approach with `TopKPooling` inside a `K3TopKNet` model, trained with the Adam optimizer for 10 epochs, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `TopKPooling` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops
import numpy as np

import k3_node
from k3_node import layers as k3_layers
from k3_node.datasets import TUDataset
from k3_node.loader import DataLoader

title = "Graph Classification with TopKPooling on Colors Dataset"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset & Loaders
try:
    dataset = TUDataset(root="./data/COLORS-3", name="COLORS-3", use_node_attr=True)
except Exception:
    dataset = TUDataset(root="./data/PROTEINS", name="PROTEINS", use_node_attr=True)

train_batch_size = 60
# drop_last=True keeps every training batch at a fixed size. Keras's
# fit()/train_on_batch() runs one internal forward pass with constant-filled
# placeholder tensors to validate the loss pipeline; if the graph count in a
# batch were only known from real data (e.g. inferred as `batch.max()+1`,
# which min_score-based TopKPooling can shrink further by dropping every
# node of a graph), that placeholder pass would infer the wrong output size
# and the validation pass would raise a spurious shape-mismatch error. With
# a fixed batch size we instead pass a static `size` to global_add_pool.
train_loader = DataLoader(dataset[:500], batch_size=train_batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(dataset[500:], batch_size=60)

in_channels = dataset.num_features

# 2. GINConv & TopKPooling Model
class K3TopKNet(keras.Model):
    def __init__(self, in_channels, batch_size):
        super().__init__()
        self.mlp1 = keras.Sequential([layers.Dense(64, activation="relu"), layers.Dense(64)])
        self.conv1 = k3_layers.GINConv(self.mlp1)
        self.pool1 = k3_layers.TopKPooling(64, min_score=0.05)
        self.mlp2 = keras.Sequential([layers.Dense(64, activation="relu"), layers.Dense(64)])
        self.conv2 = k3_layers.GINConv(self.mlp2)
        self.lin = layers.Dense(1)
        self.batch_size = batch_size

    def build(self, input_shape=None):
        self.mlp1.build((None, in_channels))
        self.conv1.build((None, in_channels))
        self.pool1.build((None, 64))
        self.mlp2.build((None, 64))
        self.conv2.build((None, 64))
        self.lin.build((None, 64))
        self.built = True

    def call(self, inputs):
        x, edge_index, batch = inputs["x"], inputs["edge_index"], inputs.get("batch", None)
        out = ops.relu(self.conv1(x, edge_index))
        out, edge_index, _, batch, perm, score = self.pool1(out, edge_index, batch=batch)
        out = ops.relu(self.conv2(out, edge_index))
        out = k3_layers.global_add_pool(out, batch, size=self.batch_size)
        return ops.squeeze(self.lin(out), axis=-1)

k3_model = K3TopKNet(in_channels, batch_size=train_batch_size)
k3_model.build()

# 3. Model Compilation
compile_kwargs = {
    "optimizer": keras.optimizers.Adam(learning_rate=0.001),
    "loss": keras.losses.MeanSquaredError(),
    "metrics": [keras.metrics.RootMeanSquaredError(name="rmse")],
}
if backend == "jax":
    compile_kwargs["jit_compile"] = False

k3_model.compile(**compile_kwargs)

# 4. Generator
def to_np(t, dtype=None):
    if t is None:
        return None
    if isinstance(t, np.ndarray):
        return t.astype(dtype) if dtype else t
    if hasattr(t, "detach"):
        t = t.detach()
    if hasattr(t, "cpu") and type(t).__module__.startswith("torch"):
        t = t.cpu()
    if hasattr(t, "numpy") and callable(t.numpy):
        t = t.numpy()
    return np.asarray(t, dtype=dtype)

def make_generator(loader):
    while True:
        for batch in loader:
            inputs = {
                "x": to_np(batch.x, dtype=np.float32),
                "edge_index": to_np(batch.edge_index, dtype=np.int32),
                "batch": to_np(batch.batch, dtype=np.int32) if hasattr(batch, "batch") and batch.batch is not None else None,
            }
            y = to_np(batch.y, dtype=np.float32).reshape(-1)
            yield inputs, y

print(f"Training K3-Node TopKPooling model on {backend} backend...")
history = k3_model.fit(
    make_generator(train_loader),
    steps_per_epoch=len(train_loader),
    epochs=10,
    verbose=1,
)

print("\n✓ K3-Node execution completed successfully!")